# 小売売上データで学ぶ時系列分析と予測

この notebook では `store_sales.csv` と `promotion_data.csv` を使って、小売売上データの読み込み、基本的な確認、季節性の可視化、STL 分解、ETS / ARIMA / Prophet / LightGBM による予測までを一通り試します。

GitHub でそのまま読めるように、コードだけでなく「なぜその処理をしているのか」も短く添えています。

## この notebook の流れ

1. データを読み込む
2. データの粒度や欠損の有無を確認する
3. 1 系列を例に季節性を観察する
4. STL で系列を分解する
5. ETS / ARIMA / Prophet / LightGBM で予測する

## 0. 準備

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

pd.options.display.max_rows = 400
pd.options.display.max_columns = None

import warnings
warnings.filterwarnings('ignore')

## 1. データの読み込み

今回使う主な列は次のとおりです。

- `store`: 店舗 ID
- `dept`: 部門 ID
- `week`: 週の起点日
- `sales`: 売上金額
- `promotion_sales`: 販促施策による売上金額

In [ ]:
df_sales = pd.read_csv('data/store_sales.csv', parse_dates=['week'])
df_sales.head(2)

In [ ]:
df_promotion = pd.read_csv('data/promotion_data.csv', parse_dates=['week'])
df_promotion.head(2)

販促データは一部の週で欠損する可能性があるため、結合後に 0 で補完します。

In [ ]:
df_all = pd.merge(
    df_sales,
    df_promotion,
    how='left',
    on=['store', 'dept', 'week']
)
df_all['promotion_sales'] = df_all['promotion_sales'].fillna(0)
df_all.head(2)

## 2. データのざっくり確認

まずはデータの件数、主キー、期間、重複、系列数を確認します。
この段階で系列の粒度が分かると、後続の特徴量設計やモデル選択がしやすくなります。

In [ ]:
df_sales.head(2), df_sales.shape

In [ ]:
primary_key = ['week', 'store', 'dept']
primary_key, (df_sales['week'].min(), df_sales['week'].max())

In [ ]:
df_sales.shape[0] == df_sales[['week', 'store', 'dept']].drop_duplicates().shape[0]

In [ ]:
df_sales[['store', 'dept']].drop_duplicates().shape[0]

各系列が週次で連続しているかも確認しておきます。
時系列モデルでは、欠けている週があるとラグ特徴量や周期の解釈に影響が出やすいためです。

In [ ]:
date_summary = df_sales.groupby(['store', 'dept'])['week'].agg(['min', 'max', 'nunique']).reset_index()
date_summary['missing_weeks'] = (date_summary['max'] - date_summary['min']).dt.days // 7 + 1 - date_summary['nunique']
date_summary[date_summary['missing_weeks'] > 0].head()

In [ ]:
date_summary[date_summary['missing_weeks'] > 0].shape[0]

## 3. 1 系列を例に時系列を眺める

ここからは `store=1`、`dept=1` を例にして、売上系列の形を確認します。
まずは単純な折れ線図で、全体の流れと大きな波をつかみます。

In [ ]:
df_sample = df_sales[(df_sales['store'] == 1) & (df_sales['dept'] == 1)].sort_values('week').copy()
plt.figure(figsize=(12, 4))
plt.plot(df_sample['week'], df_sample['sales'], '.-')
plt.xticks(rotation=90)
plt.title('Store 1 / Dept 1 Weekly Sales')
plt.show()

In [ ]:
display(df_sample.head())
df_sample.tail(2)

グラフを見ると、年間のどこかで似たような波が繰り返されていそうに見えます。
そこで次は、年ごとに週番号をそろえて季節性を確認します。

## 4. 季節性を確認する

In [ ]:
df = df_sample.copy()
df['year'] = df['week'].dt.year
df['week_of_year'] = df['week'].dt.isocalendar().week.astype(int)

for year in df['year'].unique():
    tmp_df = df[df['year'] == year]
    plt.plot(tmp_df['week_of_year'], tmp_df['sales'], '.-', label=str(year))
plt.legend()
plt.title('Seasonal Line Plot')
plt.show()

季節性の形だけでなく、同じ週番号でどの程度ばらつくかを見るには箱ひげ図が便利です。

In [ ]:
df_boxplot = df.groupby(['week_of_year'])['sales'].agg(list).reset_index()
plt.figure(figsize=(14, 5))
plt.boxplot(df_boxplot['sales'].values, labels=df_boxplot['week_of_year'].values)
plt.xticks(rotation=90)
plt.title('Seasonal Box Plot')
plt.show()

販促の影響が強そうな時期をいったん外して、年ごとの水準差を見ることもできます。

In [ ]:
df_trend = df.copy()
df_trend = df_trend[(df_trend['week_of_year'] >= 19) & (df_trend['week_of_year'] <= 43)]
trend_boxplot = df_trend.groupby(['year'])['sales'].agg(list).reset_index()
plt.boxplot(trend_boxplot['sales'].values, labels=trend_boxplot['year'].values)
plt.title('Trend Box Plot')
plt.show()

## 5. ACF で周期性を確かめる

見た目だけでなく、自己相関から周期性を確認します。
ACF（自己相関関数）は、系列とラグ付き系列の相関を見て、どの周期で似た動きが繰り返されるかを調べる道具です。

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

plot_acf(df_sample['sales'], lags=140).show()

52 週付近で相関が高ければ、週次データに年周期があると考えやすくなります。

## 6. STL で系列を分解する

STL は系列を「トレンド」「季節成分」「残差」に分けて眺めるための方法です。
予測モデルではありませんが、系列の構造を理解するうえでとても役立ちます。

In [ ]:
from statsmodels.tsa.seasonal import STL

stl = STL(df_sample['sales'].values, period=52)
stl_result = stl.fit()
stl_result.plot();

In [ ]:
df_stl = df_sample.copy()
df_stl['trend'] = stl_result.trend
df_stl['seasonal'] = stl_result.seasonal
df_stl['residual'] = stl_result.resid
df_stl.head()

## 7. ETS で 2 週間先を予測する

ETS は指数平滑法に基づくモデルで、比較的短めの系列でも試しやすいのが利点です。
ここでは加法トレンド・加法季節性を持つモデルとして学習します。

In [ ]:
from statsmodels.tsa.exponential_smoothing.ets import ETSModel

forecast_origin = pd.to_datetime('2012-07-30')
ets_train = df_sample[df_sample['week'] <= forecast_origin][['week', 'sales']].copy()
ets_train = ets_train.set_index('week')

ets_model = ETSModel(
    ets_train['sales'],
    error='add',
    trend='add',
    seasonal='add',
    seasonal_periods=52,
)
ets_fit = ets_model.fit()

In [ ]:
display(ets_train.head(2))
ets_train.tail()

In [ ]:
ets_train['fit_sales'] = ets_fit.fittedvalues
ets_plot = ets_train[(ets_train.index >= '2011-01-03') & (ets_train.index <= '2011-06-27')]
plt.plot(ets_plot['sales'], '.-', label='actual')
plt.plot(ets_plot['fit_sales'], '.-', label='fit')
plt.legend()
plt.xticks(rotation=90)
plt.show()

In [ ]:
h = 2
ets_forecast = ets_fit.forecast(steps=h)
ets_result = pd.DataFrame({
    'week': pd.date_range(start=ets_train.index.max() + pd.Timedelta(weeks=1), periods=h, freq='W-MON'),
    'pred_sales': ets_forecast.values,
})
ets_test = df_sample[(df_sample['week'] > forecast_origin) & (df_sample['week'] <= forecast_origin + pd.Timedelta(weeks=2))][['week', 'sales']]
ets_result.merge(ets_test, on='week', how='left')

In [ ]:
ets_result

## 8. ARIMA で 2 週間先を予測する

ARIMA は自己回帰、差分、移動平均を組み合わせる古典的な時系列モデルです。
まずは原系列と差分系列の自己相関を見て、差分の必要性を確認します。

In [ ]:
plot_acf(df_sample['sales'], lags=60).show()

df_arima = df_sample.copy()
df_arima['sales_lag1'] = df_arima['sales'].shift(1)
df_arima['sales_diff'] = df_arima['sales'] - df_arima['sales_lag1']
plot_acf(df_arima['sales_diff'].dropna(), lags=60).show()
plot_pacf(df_arima['sales_diff'].dropna(), lags=30).show()

In [ ]:
df_arima[['week', 'sales', 'sales_diff']].head()

In [ ]:
from statsmodels.tsa.arima.model import ARIMA

arima_train = df_sample[df_sample['week'] <= forecast_origin][['week', 'sales']].copy()
arima_train = arima_train.set_index('week')

p = 1
d = 1
q = 0

arima_model = ARIMA(arima_train['sales'], order=(p, d, q))
arima_fit = arima_model.fit()
arima_fit.summary()

In [ ]:
display(arima_train.head(2))
arima_train.tail()

In [ ]:
arima_forecast = arima_fit.get_forecast(steps=2)
arima_result = pd.DataFrame({
    'week': pd.date_range(start=arima_train.index.max() + pd.Timedelta(weeks=1), periods=2, freq='W-MON'),
    'pred_sales': arima_forecast.predicted_mean.values,
})
arima_test = df_sample[(df_sample['week'] > forecast_origin) & (df_sample['week'] <= forecast_origin + pd.Timedelta(weeks=2))][['week', 'sales']]
arima_result.merge(arima_test, on='week', how='left')

In [ ]:
arima_result

## 9. Prophet で販促情報も使って予測する

Prophet はトレンド、季節性、外生変数を比較的扱いやすいモデルです。
ここでは `promotion_sales` を追加の説明変数として使います。

In [ ]:
import prophet

In [ ]:
prophet_train = df_all[(df_all['week'] <= '2012-07-30') & (df_all['store'] == 1) & (df_all['dept'] == 1)].copy()
prophet_train = prophet_train.rename(columns={'week': 'ds', 'sales': 'y'})

prophet_test = df_all[(df_all['week'] == '2012-08-06') & (df_all['store'] == 1) & (df_all['dept'] == 1)].copy()
prophet_test = prophet_test.rename(columns={'week': 'ds', 'sales': 'y'})

prophet_model = prophet.Prophet(yearly_seasonality=True)
prophet_model.add_regressor('promotion_sales')
prophet_model.fit(prophet_train)

In [ ]:
display(prophet_train[['ds', 'y', 'promotion_sales']].tail())
prophet_test[['ds', 'y', 'promotion_sales']]

In [ ]:
prophet_fit = prophet_model.predict(prophet_train)
prophet_model.plot_components(prophet_fit)
prophet_model.plot(prophet_fit);

In [ ]:
prophet_pred = prophet_model.predict(prophet_test)
prophet_test['yhat'] = prophet_pred['yhat'].values
prophet_test[['ds', 'y', 'promotion_sales', 'yhat']]

同じ考え方で、1 店舗内のすべての部門をまとめて順番に予測することもできます。

In [ ]:
dept_list = df_all[df_all['store'] == 1]['dept'].unique()

all_prophet_result = []
for dept in dept_list:
    train_part = df_all[(df_all['week'] <= '2012-07-30') & (df_all['store'] == 1) & (df_all['dept'] == dept)].copy()
    train_part = train_part.rename(columns={'week': 'ds', 'sales': 'y'})

    test_part = df_all[(df_all['week'] > '2012-07-30') & (df_all['week'] <= '2012-08-06') & (df_all['store'] == 1) & (df_all['dept'] == dept)].copy()
    test_part = test_part.rename(columns={'week': 'ds', 'sales': 'y'})

    if (train_part.shape[0] > 100) and (test_part.shape[0] > 0):
        model = prophet.Prophet(yearly_seasonality=True)
        model.add_regressor('promotion_sales')
        model.fit(train_part)

        pred_part = model.predict(test_part)
        test_part['yhat'] = pred_part['yhat'].values
        all_prophet_result.append(test_part)

all_prophet_result = pd.concat(all_prophet_result)
all_prophet_result.head()

## 10. LightGBM で特徴量ベースの予測を行う

LightGBM では、時系列をそのまま入れるのではなく、ラグ特徴量や前年同週の値を特徴量として作って学習します。
ここでは最小限の特徴量として、前週・前年同週・当週の販促売上を使います。

In [ ]:
import lightgbm as lgb
from lightgbm.sklearn import LGBMRegressor

In [ ]:
lgb_sample = df_all[(df_all['store'] == 1) & (df_all['dept'] == 1)].sort_values('week').copy()
feature_cols = []

lgb_sample['sales_lw'] = lgb_sample['sales'].shift(1)
lgb_sample['promotion_lw'] = lgb_sample['promotion_sales'].shift(1)
feature_cols = feature_cols + ['sales_lw', 'promotion_lw']

lgb_sample['sales_ly'] = lgb_sample['sales'].shift(52)
lgb_sample['promotion_ly'] = lgb_sample['promotion_sales'].shift(52)
feature_cols = feature_cols + ['sales_ly', 'promotion_ly']

feature_cols = feature_cols + ['promotion_sales']

for col in feature_cols:
    lgb_sample = lgb_sample[~lgb_sample[col].isna()]

x_train = lgb_sample[lgb_sample['week'] <= '2012-07-30'][feature_cols].values
y_train = lgb_sample[lgb_sample['week'] <= '2012-07-30']['sales'].values
x_test = lgb_sample[lgb_sample['week'] == '2012-08-06'][feature_cols].values
y_test = lgb_sample[lgb_sample['week'] == '2012-08-06']['sales'].values

In [ ]:
display(lgb_sample[['week'] + feature_cols + ['sales']].head())
lgb_sample.shape

In [ ]:
lgb_model = LGBMRegressor()
lgb_model.fit(x_train, y_train)
y_pred = lgb_model.predict(x_test)
y_pred, y_test

複数部門をまとめて学習する場合は、部門ごとにラグを計算するのがポイントです。

In [ ]:
lgb_all = df_all[df_all['store'] == 1].sort_values(['dept', 'week']).copy()
feature_cols = []

lgb_all['sales_lw'] = lgb_all.groupby(['dept'])['sales'].shift(1)
lgb_all['promotion_lw'] = lgb_all.groupby(['dept'])['promotion_sales'].shift(1)
feature_cols = feature_cols + ['sales_lw', 'promotion_lw']

lgb_all['sales_ly'] = lgb_all.groupby(['dept'])['sales'].shift(52)
lgb_all['promotion_ly'] = lgb_all.groupby(['dept'])['promotion_sales'].shift(52)
feature_cols = feature_cols + ['sales_ly', 'promotion_ly']

feature_cols = feature_cols + ['promotion_sales']

for col in feature_cols:
    lgb_all = lgb_all[~lgb_all[col].isna()]

x_train = lgb_all[lgb_all['week'] <= '2012-07-30'][feature_cols].values
y_train = lgb_all[lgb_all['week'] <= '2012-07-30']['sales'].values
x_test = lgb_all[lgb_all['week'] == '2012-08-06'][feature_cols].values
y_test = lgb_all[lgb_all['week'] == '2012-08-06']['sales'].values

lgb_model = LGBMRegressor()
lgb_model.fit(x_train, y_train)
y_pred = lgb_model.predict(x_test)
y_pred, y_test

In [ ]:
display(lgb_all[['dept', 'week'] + feature_cols + ['sales']].head())
lgb_all.shape

## 11. 使い分けの目安

- STL: 予測というより、系列の構造理解に向いています。
- ETS: 比較的短い系列でも試しやすいベースラインです。
- ARIMA: 差分後の自己相関構造を使いたいときに有効です。
- Prophet: トレンドや季節性を見ながら、外生変数も素直に入れたいときに便利です。
- LightGBM: 系列数が多く、ラグ特徴量や外生変数をまとめて扱いたいときに強みがあります。

## まとめ

この notebook では、小売売上データを題材に、探索的な確認から複数の予測手法までを通しで確認しました。
実務では、まず系列の粒度や欠損の有無を確かめ、そのうえで STL や可視化で構造をつかみ、最後に予測モデルを比較する流れが扱いやすいです。